In [1]:
import pandas as pd
import numpy as np
from MyTfIdfVectorizer import MyTfIdfVectorizer 
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import re


# Part E: Kiểm thử  trên corpus nhỏ và so sánh với TfidfVectorizer của sklearn

In [2]:
data = pd.DataFrame({
    "text": [
        "cat eats fish",
        "dog eats fish",
        "cat likes fish"
    ]
})

my_vectorizer = MyTfIdfVectorizer(data)
sklearn_vectorizer = TfidfVectorizer()

## 0.1 My Tf-Idf

In [3]:
my_vectorizer.buildVocabulary("text")
expected_vocab = [
    "cat",
    "dog",
    "eats",
    "fish",
    "likes"
]

assert my_vectorizer.vocab == expected_vocab


In [4]:
my_vectorizer.compute_counts("text")

expected_counts = np.array([
    [1, 0, 1, 1, 0],
    [0, 1, 1, 1, 0],
    [1, 0, 0, 1, 1]
], dtype=np.float32)

assert np.array_equal(
    my_vectorizer.count_matrix.toarray(),
    expected_counts
)

In [5]:
# compute_tf
my_vectorizer.compute_tf()

expected_tf = np.array([
    [1/3, 0,   1/3, 1/3, 0],
    [0,   1/3, 1/3, 1/3, 0],
    [1/3, 0,   0,   1/3, 1/3]
], dtype=np.float32)

assert np.allclose(
    my_vectorizer.tf.toarray(),
    expected_tf,
    atol=1e-9
)

In [6]:
# compute_df
my_vectorizer.compute_df()

expected_df = np.array([2, 1, 2, 3, 1])

assert np.array_equal(
    my_vectorizer.df,
    expected_df
)


In [7]:
# compute idf
my_vectorizer.compute_idf()

expected_idf = np.log(
    4 / (1 + expected_df)
) + 1

assert np.allclose(
    my_vectorizer.idf,
    expected_idf,
    atol=1e-9
)


In [8]:
# compute_tfidf

my_vectorizer.compute_tfidf()

raw_tfidf = expected_tf * expected_idf

norms = np.linalg.norm(
    raw_tfidf,
    axis=1,
    keepdims=True
)

expected_tfidf = raw_tfidf / norms
print(expected_tfidf)
print(my_vectorizer.tfidf)
assert np.allclose(
    my_vectorizer.tfidf.toarray(),
    expected_tfidf,
    atol=1e-9
)

[[0.61980538 0.         0.61980538 0.48133417 0.        ]
 [0.         0.72033345 0.54783215 0.42544054 0.        ]
 [0.54783215 0.         0.         0.42544054 0.72033345]]
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9 stored elements and shape (3, 5)>
  Coords	Values
  (0, 3)	0.4813341733388614
  (0, 2)	0.6198053781535761
  (0, 0)	0.6198053781535761
  (1, 3)	0.425440540311846
  (1, 2)	0.5478321498361728
  (1, 1)	0.7203334521352189
  (2, 4)	0.7203334521352189
  (2, 3)	0.425440540311846
  (2, 0)	0.5478321498361728


In [9]:
#compute_cosine_similarity
my_vectorizer.compute_cosine_similarity()

# Test diagonal = 1
assert np.allclose(
    np.diag(my_vectorizer.cosine_similarity),
    np.ones(3),
    atol=1e-9
)
print(my_vectorizer.cosine_similarity)
# Test symmetry
assert np.allclose(
    my_vectorizer.cosine_similarity,
    my_vectorizer.cosine_similarity.T,
    atol=1e-9
)


Check running
[[1.         0.54432838 0.54432838]
 [0.54432838 1.         0.18099965]
 [0.54432838 0.18099965 1.        ]]


## 0.2 Compare with sklearn TfIdfVectorizer

In [10]:
sklearn_vectorizer = TfidfVectorizer(
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False
)
cv = CountVectorizer()
count_matrix = cv.fit_transform(data["text"])

row_sums = np.asarray(count_matrix.sum(axis=1)).ravel()

sklearn_tf = count_matrix.multiply(
    1 / row_sums[:, None]
)
sklearn_tfidf = sklearn_vectorizer.fit_transform(data["text"])
sklearn_idf = sklearn_vectorizer.idf_

print((my_vectorizer.tf))
print((sklearn_tf))
print(type(my_vectorizer.idf))
print(type(my_vectorizer.tfidf))


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9 stored elements and shape (3, 5)>
  Coords	Values
  (0, 3)	0.3333333333333333
  (0, 2)	0.3333333333333333
  (0, 0)	0.3333333333333333
  (1, 3)	0.3333333333333333
  (1, 2)	0.3333333333333333
  (1, 1)	0.3333333333333333
  (2, 4)	0.3333333333333333
  (2, 3)	0.3333333333333333
  (2, 0)	0.3333333333333333
<COOrdinate sparse matrix of dtype 'float64'
	with 9 stored elements and shape (3, 5)>
  Coords	Values
  (0, 0)	0.3333333333333333
  (0, 2)	0.3333333333333333
  (0, 3)	0.3333333333333333
  (1, 2)	0.3333333333333333
  (1, 3)	0.3333333333333333
  (1, 1)	0.3333333333333333
  (2, 0)	0.3333333333333333
  (2, 3)	0.3333333333333333
  (2, 4)	0.3333333333333333
<class 'numpy.ndarray'>
<class 'scipy.sparse._csr.csr_matrix'>


In [11]:
def compare_matrices(name, sklearn_result, my_result, atol=1e-6):

    if hasattr(sklearn_result, "toarray"):
        sklearn_result = sklearn_result.toarray()

    print(f"{name}:")
    print("  Shape:", sklearn_result.shape, my_result.shape)

    print(
        "  Equal:",
        np.allclose(
            sklearn_result,
            my_result,
            atol=atol
        )
    )

    print(
        "  Max error:",
        np.max(
            np.abs(
                sklearn_result - my_result
            )
        )
    )

In [12]:
compare_matrices(
    "TF",
    sklearn_tf,
    my_vectorizer.tf.toarray()
)

compare_matrices(
    "IDF",
    sklearn_idf,
    my_vectorizer.idf
)

compare_matrices(
    "TF-IDF",
    sklearn_tfidf,
    my_vectorizer.tfidf.toarray()
)


TF:
  Shape: (3, 5) (3, 5)
  Equal: True
  Max error: 0.0
IDF:
  Shape: (5,) (5,)
  Equal: True
  Max error: 1.602477883722031e-08
TF-IDF:
  Shape: (3, 5) (3, 5)
  Equal: True
  Max error: 5.0912635218836044e-09


# Part F: Preprocessing Ablation

In [13]:
df = pd.read_json("/home/vitquay1708/Study_Space/NLP/lab1/c4-train.00000-of-01024-30K.json.gz", lines=True)
df.head(5)

,text,timestamp,url
0,Beginners BBQ Class Taking Place in Missoula!\...,2019-04-25 12:57:54+00:00,https://klyq.com/beginners-bbq-class-taking-pl...
1,Discussion in 'Mac OS X Lion (10.7)' started b...,2019-04-21 10:07:13+00:00,https://forums.macrumors.com/threads/restore-f...
2,Foil plaid lycra and spandex shortall with met...,2019-04-25 10:40:23+00:00,https://awishcometrue.com/Catalogs/Clearance/T...
3,How many backlinks per day for new site?\nDisc...,2019-04-21 12:46:19+00:00,https://www.blackhatworld.com/seo/how-many-bac...
4,The Denver Board of Education opened the 2017-...,2019-04-20 14:33:21+00:00,http://bond.dpsk12.org/category/news/


## 1. Minimal

In [14]:
df['text_lower'] = df['text'].str.lower()
df['pipeline_1'] = df['text_lower'].apply(lambda x: x.split())
display(df[['text', 'text_lower', 'pipeline_1']].head(3))


,text,text_lower,pipeline_1
0,Beginners BBQ Class Taking Place in Missoula!\...,beginners bbq class taking place in missoula!\...,"[beginners, bbq, class, taking, place, in, mis..."
1,Discussion in 'Mac OS X Lion (10.7)' started b...,discussion in 'mac os x lion (10.7)' started b...,"[discussion, in, 'mac, os, x, lion, (10.7)', s..."
2,Foil plaid lycra and spandex shortall with met...,foil plaid lycra and spandex shortall with met...,"[foil, plaid, lycra, and, spandex, shortall, w..."


## 2. Pipeline B - Normalized

In [15]:
df['pipeline_2'] = df['text_lower'].apply(lambda x: re.findall(r'\b\w+\b', x))
display(df[['text', 'text_lower', 'pipeline_2']].head(3))

,text,text_lower,pipeline_2
0,Beginners BBQ Class Taking Place in Missoula!\...,beginners bbq class taking place in missoula!\...,"[beginners, bbq, class, taking, place, in, mis..."
1,Discussion in 'Mac OS X Lion (10.7)' started b...,discussion in 'mac os x lion (10.7)' started b...,"[discussion, in, mac, os, x, lion, 10, 7, star..."
2,Foil plaid lycra and spandex shortall with met...,foil plaid lycra and spandex shortall with met...,"[foil, plaid, lycra, and, spandex, shortall, w..."


## 3 Pipeline C - Extended

In [16]:
from transformers import AutoTokenizer

subword_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

df['pipeline_3'] = df['text_lower'].apply(lambda x: subword_tokenizer.tokenize(x))
display(df[['text', 'text_lower', 'pipeline_3']].head(3))

/home/vitquay1708/miniconda3/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2531 > 512). Running this sequence through the model will result in indexing errors


,text,text_lower,pipeline_3
0,Beginners BBQ Class Taking Place in Missoula!\...,beginners bbq class taking place in missoula!\...,"[begin, ##ners, bb, ##q, class, taking, place,..."
1,Discussion in 'Mac OS X Lion (10.7)' started b...,discussion in 'mac os x lion (10.7)' started b...,"[discussion, in, ', mac, os, x, lion, (, 10, ...."
2,Foil plaid lycra and spandex shortall with met...,foil plaid lycra and spandex shortall with met...,"[foil, plaid, l, ##y, ##cr, ##a, and, span, ##..."


## Result Comparison

In [17]:
from collections import Counter

def tokenizer1(text):
    return text.split()

def tokenizer2(text):
    return re.findall(r'\b\w+\b', text)

def tokenizer3(text):
    return subword_tokenizer.tokenize(text)

tokenizers = {'pipeline_1':tokenizer1, 'pipeline_2':tokenizer2, 'pipeline_3':tokenizer3}
def calculate_oov_rate(tokenized_docs, vocab):
    total = 0
    oov = 0
    for doc in tokenized_docs:
        for token in doc:
            total += 1
            if token not in vocab:
                oov+=1
    return oov/total if total > 0 else 0

def compare_pipelines(
    df,
    pipeline_columns,
    test_docs,
    tokenizers
):
    results = {}

    for _, column in enumerate(pipeline_columns):

        docs = df[column].dropna().tolist()
        vocabulary = set(
            token
            for doc in docs
            for token in doc
        )
        vocab_size = len(vocabulary)
        avg_tokens = (
            sum(len(doc) for doc in docs) / len(docs)
            if docs else 0
        )
        n_docs = len(docs)

        non_zero = sum(
            len(set(doc))
            for doc in docs
        )

        total_entries = n_docs * vocab_size

        sparsity = (
            1 - non_zero / total_entries
            if total_entries > 0
            else 0
        )

        test_tokenized = [
            tokenizers[column](text)
            for text in test_docs
        ]

        oov_rate = calculate_oov_rate(test_tokenized, vocabulary)

        results[column] = {
            "Vocabulary size": vocab_size,
            "Average tokens/document": avg_tokens,
            "Matrix sparsity": sparsity,
            "OOV rate": oov_rate
        }

    return pd.DataFrame(results).T

In [18]:
test_docs = [
    "The cat is running quickly.",
    "I love machine learning.",
    "This is an amazing example.",
    "Natural language processing is interesting.",
    "The model generates new tokens.",
    "Students are learning artificial intelligence.",
    "The weather is beautiful today.",
    "This sentence contains an unknownword.",
    "Transformers are widely used in NLP.",
    "Subword tokenization handles rare words."
]
result_comparison = compare_pipelines(df, ['pipeline_1', 'pipeline_2', 'pipeline_3'],test_docs,tokenizers)

display(result_comparison)

,Vocabulary size,Average tokens/document,Matrix sparsity,OOV rate
pipeline_1,473388.0,361.089067,0.999611,0.24
pipeline_2,193837.0,369.696367,0.999122,0.24
pipeline_3,28339.0,465.099200,0.993200,0.00


# Part G: Document Search Engine

## 1. Building TF-IDF index on 30K documents

In [20]:
import time
start_time = time.time()

search_engine = MyTfIdfVectorizer(df)
search_engine.fit("text")
elapsed = time.time() - start_time
print(f"Done in {elapsed:.2f}s")
print(f"Documents: {search_engine.tfidf.shape[0]}")
print(f"Vocabulary size: {len(search_engine.vocab)}")

Done in 19.84s
Documents: 30000
Vocabulary size: 254766


## 2. Queries

In [36]:
user_query = input("Enter your search query: ")
results = search_engine.search(user_query, top_k=5)
print(f'\nResults for: "{user_query}"')
display(results)





Results for: ""transformer language model""


,Rank,Document ID,Similarity,Document Preview
0,1,27936,0.2913,"hi, I am having problems with transformer / ci..."
1,2,25428,0.2811,"Note: If you're on an iPhone, you cannot chang..."
2,3,4075,0.2210,Looking for Spanish language instructor to imp...
3,4,701,0.2175,Program in Teaching French as a Foreign Langua...
4,5,13690,0.2148,Text in finnish language missing. Sorry for th...


## Using another tokenizer

In [30]:
start_time = time.time()
search_engine_bert = MyTfIdfVectorizer(df, subword_tokenizer.tokenize)
search_engine_bert.fit("text")
elapsed = time.time() - start_time
print(f"Done in {elapsed:.2f}s")
print(f"Documents: {search_engine_bert.tfidf.shape[0]}")
print(f"Vocabulary size: {len(search_engine_bert.vocab)}")


Done in 218.58s
Documents: 30000
Vocabulary size: 28339


In [29]:
print(len(search_engine_bert.vocab))

28339


In [37]:
user_query = input("Enter your search query: ")
results = search_engine_bert.search(user_query, top_k=5)
print(f'\nResults for: "{user_query}"')
display(results)


Results for: ""transformer language model""


,Rank,Document ID,Similarity,Document Preview
0,1,26409,0.4412,"Les légendes de la soul, cette musique qui mêl..."
1,2,12989,0.4273,"Do you have ideas for warm-ups, seating arrang..."
2,3,17656,0.4199,"adj. causing fear or dread or terror; ""the awf..."
3,4,11816,0.4187,"""Keeping up with friends is faster than ever. ..."
4,5,14427,0.3816,"http://www.JewishWorldReview.com | At first, t..."
